# 02 — Baselines

Este notebook documenta os modelos de referência utilizados no projeto.

Os baselines são importantes para verificar se as abordagens posteriores realmente acrescentam desempenho em relação a soluções simples.

São considerados:

1. classe majoritária;
2. TF-IDF + Regressão Logística em holdout;
3. TF-IDF + Regressão Logística em validação cruzada;
4. Grid Search do baseline TF-IDF + Regressão Logística.

A implementação utilizada aqui é a mesma de `experiments/baselines.py`, evitando manter uma segunda versão do código dos baselines dentro do notebook.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

CWD = Path.cwd().resolve()

if (CWD / "src").exists():
    ROOT = CWD
elif (CWD.parent / "src").exists():
    ROOT = CWD.parent
else:
    raise RuntimeError(
        "Não foi possível localizar a raiz do projeto. "
        "Execute o notebook dentro do repositório ep1-pln."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import carregar_dados_treino

from experiments.baselines import (
    executar_baseline_majoritario,
    executar_baseline_tfidf_holdout,
    executar_baseline_tfidf_cv,
    executar_grid_tfidf_baseline,
)


In [ ]:
X, y = carregar_dados_treino()

print("Quantidade de exemplos:", len(X))
print("\nDistribuição das classes:")
print(y.value_counts())


## 1. Baseline de classe majoritária

Este baseline ignora o texto e prevê sempre a classe mais frequente no conjunto de treinamento.

Ele fornece uma referência mínima para a tarefa.


In [ ]:
resultado_majoritario = executar_baseline_majoritario(
    X,
    y,
)

resultado_majoritario


## 2. Baseline oficial — TF-IDF + Regressão Logística em holdout

O segundo baseline utiliza representação TF-IDF de palavras seguida de Regressão Logística.

A transformação textual é ajustada apenas com os dados de treinamento da divisão, evitando vazamento de informação.


In [ ]:
resultado_holdout = executar_baseline_tfidf_holdout(
    X,
    y,
)

resultado_holdout


## 3. Baseline com validação cruzada

Para obter uma estimativa menos dependente de uma única divisão dos dados, o mesmo baseline é avaliado com validação cruzada estratificada de 5 folds.


In [ ]:
resultado_cv, folds_cv = executar_baseline_tfidf_cv(
    X,
    y,
)

folds_cv


In [ ]:
pd.DataFrame(
    {
        "metrica": [
            "Accuracy",
            "Macro-F1",
            "Accuracy treino",
            "Gap treino-validação",
        ],
        "valor": [
            resultado_cv["accuracy"],
            resultado_cv["f1_macro"],
            resultado_cv["train_accuracy"],
            resultado_cv["accuracy_gap"],
        ],
    }
)


## 4. Grid Search do baseline

Também foi realizado um Grid Search simples sobre o baseline, variando o intervalo de n-gramas e o parâmetro `C` da Regressão Logística.

Esse experimento continua sendo considerado baseline, porque utiliza a mesma família tradicional TF-IDF + Regressão Logística.


In [ ]:
resultado_grid, tabela_grid = executar_grid_tfidf_baseline(
    X,
    y,
)

tabela_grid


## 5. Comparação dos baselines


In [ ]:
comparacao = pd.DataFrame(
    [
        resultado_majoritario,
        resultado_holdout,
        resultado_cv,
        resultado_grid,
    ]
)

colunas = [
    "experiment_name",
    "representation",
    "model",
    "evaluation",
    "accuracy",
    "accuracy_std",
    "f1_macro",
    "train_accuracy",
    "accuracy_gap",
]

comparacao[
    [
        coluna
        for coluna in colunas
        if coluna in comparacao.columns
    ]
]


## Resultados de referência do projeto

Nos resultados consolidados do projeto, os principais valores observados foram aproximadamente:

- classe majoritária: **0,3431 de Accuracy**;
- TF-IDF + Regressão Logística em 5-fold: **0,4572 de Accuracy** e **0,4557 de Macro-F1**;
- Grid Search do baseline: **0,4613 de Accuracy** e **0,4602 de Macro-F1**.

Esses valores foram utilizados como referência para os experimentos posteriores.


## Relação com o modelo final

Os baselines deste notebook não correspondem ao modelo utilizado para gerar a entrega final.

Após os experimentos adicionais, o modelo selecionado foi o **Ensemble TF-IDF + Sentence Transformer**, implementado em `experiments/ensemble.py`.

A confirmação em 5 folds apresentou aproximadamente:

- **Accuracy: 0,4625**
- **Macro-F1: 0,4611**

A geração dos rótulos do conjunto de teste é feita por:

```bash
python -m experiments.predict_test
```

Portanto, este notebook deve ser interpretado como documentação das referências iniciais do projeto, e não como implementação do modelo final.
